In [ ]:
# ==============================================================================
# 0. Setup and Data Preparation
# ==============================================================================
# Download and unzip data in Google Colab environment
!wget https://download.pytorch.org/tutorial/data.zip -q
!unzip -q data.zip

# ------------------------------------------------------------------------------
# 1. Import necessary libraries
# ------------------------------------------------------------------------------
from __future__ import unicode_literals, print_function, division
from io import open
import glob
import os
import unicodedata
import string
import random
import time
import math

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

print("🔥 Libraries imported successfully!")

# ------------------------------------------------------------------------------
# 2. Data Preprocessing
# ------------------------------------------------------------------------------

# Define allowed character set (alphabet + special characters)
all_letters = string.ascii_letters + " .,;'"
n_letters = len(all_letters)

# Function to convert Unicode strings to pure ASCII
def unicode_to_ascii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
        and c in all_letters
    )

# Create a dictionary to hold name lists for each language and a list of all languages
category_lines = {}
all_categories = []

# Function to read a file and return it line by line
def readLines(filename):
    lines = open(filename, encoding='utf-8').read().strip().split('\n')
    return [unicode_to_ascii(line) for line in lines]

# Load data by iterating over all .txt files in the data/names folder
for filename in glob.glob('data/names/*.txt'):
    category = os.path.splitext(os.path.basename(filename))[0]
    all_categories.append(category)
    lines = readLines(filename)
    category_lines[category] = lines

n_categories = len(all_categories)

print(f"Found {n_categories} languages in total!")
print(f"Italian name sample: {category_lines['Italian'][:5]}")
print("-" * 30)

# -- Functions to convert text to tensors --

# Convert a character to an index (e.g. "a" = 0)
def letter_to_index(letter):
    return all_letters.find(letter)

# Convert a single character to a one-hot tensor of size <1 x n_letters>
def letter_to_tensor(letter):
    tensor = torch.zeros(1, n_letters)
    tensor[0][letter_to_index(letter)] = 1
    return tensor

# Convert a name (line) to a tensor of size <line_length x 1 x n_letters>
def line_to_tensor(line):
    tensor = torch.zeros(len(line), 1, n_letters)
    for i, letter in enumerate(line):
        tensor[i][0][letter_to_index(letter)] = 1
    return tensor

print("One-hot tensor of J:")
print(letter_to_tensor('J'))
print("\nTensor size of the entire Jackson:", line_to_tensor('Jackson').size())
print("✅ Data preprocessing and tensor conversion function preparation complete!")

# ------------------------------------------------------------------------------
# 3. Define the RNN Model
# ------------------------------------------------------------------------------
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(RNN, self).__init__()
        
        self.hidden_size = hidden_size
        
        # Combines input and hidden state to create the next hidden state and output.
        self.i2h = nn.Linear(input_size + hidden_size, hidden_size)
        self.i2o = nn.Linear(input_size + hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, input_tensor, hidden_tensor):
        combined = torch.cat((input_tensor, hidden_tensor), 1)
        hidden = self.i2h(combined)
        output = self.i2o(combined)
        output = self.softmax(output)
        return output, hidden

    def initHidden(self):
        # At the start of learning, the first hidden state is initialized to 0.
        return torch.zeros(1, self.hidden_size)

print("🧠 RNN model class definition complete!")

# ------------------------------------------------------------------------------
# 4. Prepare for Learning (Helper Functions, Create Model Instance)
# ------------------------------------------------------------------------------

# Function to find the most likely language from the model's output (probability)
def category_from_output(output):
    top_n, top_i = output.topk(1)
    category_i = top_i[0].item()
    return all_categories[category_i], category_i

# Function to generate random data pairs to use for learning
def random_training_example():
    category = random.choice(all_categories)
    line = random.choice(category_lines[category])
    category_tensor = torch.tensor([all_categories.index(category)], dtype=torch.long)
    line_tensor = line_to_tensor(line)
    return category, line, category_tensor, line_tensor

# Hyperparameter and model related settings
n_hidden = 128
learning_rate = 0.005

rnn = RNN(n_letters, n_hidden, n_categories)
criterion = nn.NLLLoss()
optimizer = torch.optim.SGD(rnn.parameters(), lr=learning_rate)

print("🛠️ Learning related settings complete!")

# ------------------------------------------------------------------------------
# 5. Model Learning
# ------------------------------------------------------------------------------

# Function to perform one step of learning
def train(category_tensor, line_tensor):
    hidden = rnn.initHidden()
    optimizer.zero_grad()
    
    for i in range(line_tensor.size()[0]):
        output, hidden = rnn(line_tensor[i], hidden)
    
    loss = criterion(output, category_tensor)
    loss.backward()
    optimizer.step()
    
    return output, loss.item()

# Overall learning loop
n_iters = 100000
print_every = 5000
plot_every = 1000

current_loss = 0
all_losses = []

def time_since(since):
    now = time.time()
    s = now - since
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

start = time.time()
print("\n🚀 Starting model training... (approximately 2-3 minutes)")

for iter in range(1, n_iters + 1):
    category, line, category_tensor, line_tensor = random_training_example()
    output, loss = train(category_tensor, line_tensor)
    current_loss += loss

    if iter % print_every == 0:
        guess, guess_i = category_from_output(output)
        correct = '✓' if guess == category else f'✗ ({category})'
        print(f'{iter} {iter / n_iters * 100:.0f}% ({time_since(start)}) {loss:.4f} {line} / {guess} {correct}')

    if iter % plot_every == 0:
        all_losses.append(current_loss / plot_every)
        current_loss = 0

print("✅ Model training complete!")

# Visualize learning loss
plt.figure()
plt.plot(all_losses)
plt.title('Training Loss')
plt.xlabel('Iterations (x1000)')
plt.ylabel('Loss')
plt.show()

# ------------------------------------------------------------------------------
# 6. Model Evaluation (Confusion Matrix)
# ------------------------------------------------------------------------------
print("\n📊 Evaluating model performance (creating confusion matrix)...")

# Helper function for evaluation (no learning process)
def evaluate(line_tensor):
    hidden = rnn.initHidden()
    for i in range(line_tensor.size()[0]):
        output, hidden = rnn(line_tensor[i], hidden)
    return output

# Create confusion matrix
with torch.no_grad():
    confusion = torch.zeros(n_categories, n_categories)
    n_confusion = 10000

    for i in range(n_confusion):
        category, line, category_tensor, line_tensor = random_training_example()
        output = evaluate(line_tensor)
        guess, guess_i = category_from_output(output)
        category_i = all_categories.index(category)
        confusion[category_i][guess_i] += 1

    # Normalize by dividing every row by its sum
    for i in range(n_categories):
        confusion[i] = confusion[i] / confusion[i].sum()

    # Set up plot
    fig = plt.figure(figsize=(8,8))
    ax = fig.add_subplot(111)
    cax = ax.matshow(confusion.numpy())
    fig.colorbar(cax)

    # Set up axes
    ax.set_xticklabels([''] + all_categories, rotation=90)
    ax.set_yticklabels([''] + all_categories)

    # Force label at every tick
    ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(1))

    # sphinx_gallery_thumbnail_number = 2
    plt.show()

# ------------------------------------------------------------------------------
# 7. Predicting with a new name
# ------------------------------------------------------------------------------

def predict(input_line, n_predictions=3):
    print(f'\n> {input_line}')
    with torch.no_grad():
        output = evaluate(line_to_tensor(input_line))

        # Get top N categories
        topv, topi = output.topk(n_predictions, 1, True)
        
        for i in range(n_predictions):
            value = torch.exp(topv[0][i]).item() # Convert LogSoftmax output to probability
            category_index = topi[0][i].item()
            print(f'({value:.2%}) {all_categories[category_index]}')

print("\n✨ Let's predict the nationality with a new name.")
predict('Sonesingdara') # Example of a Laotian name
predict('Kim')         # Example of a Korean name
predict('Jackson')     # Example of an English name
predict('Satoshi')     # Example of a Japanese name
predict('Schneider')   # Example of a

In [ ]:
# ==============================================================================
# 0. 환경 설정 및 데이터 준비
# ==============================================================================
# Google Colab 환경에서 데이터 다운로드 및 압축 해제
!wget https://download.pytorch.org/tutorial/data.zip -q
!unzip -q data.zip

# ------------------------------------------------------------------------------
# 1. 필요한 라이브러리 임포트
# ------------------------------------------------------------------------------
from __future__ import unicode_literals, print_function, division
from io import open
import glob
import os
import unicodedata
import string
import random
import time
import math

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

print("🔥 라이브러리 임포트 완료!")

# ------------------------------------------------------------------------------
# 2. 데이터 전처리
# ------------------------------------------------------------------------------

# 허용할 문자 집합 정의 (알파벳 + 특수문자)
all_letters = string.ascii_letters + " .,;'"
n_letters = len(all_letters)

# 유니코드 문자열을 순수 ASCII로 변환하는 함수
def unicode_to_ascii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
        and c in all_letters
    )

# 각 언어별 이름 목록을 담을 딕셔너리와 전체 언어 리스트 생성
category_lines = {}
all_categories = []

# 파일을 읽고 한 줄씩 반환하는 함수
def readLines(filename):
    lines = open(filename, encoding='utf-8').read().strip().split('\n')
    return [unicode_to_ascii(line) for line in lines]

# data/names 폴더의 모든 .txt 파일을 순회하며 데이터 로드
for filename in glob.glob('data/names/*.txt'):
    category = os.path.splitext(os.path.basename(filename))[0]
    all_categories.append(category)
    lines = readLines(filename)
    category_lines[category] = lines

n_categories = len(all_categories)

print(f"총 {n_categories}개의 언어 발견!")
print(f"Italian 이름 샘플: {category_lines['Italian'][:5]}")
print("-" * 30)

# -- 텍스트를 텐서로 변환하는 함수들 --

# 문자를 인덱스로 변환 (e.g. "a" = 0)
def letter_to_index(letter):
    return all_letters.find(letter)

# 하나의 문자를 <1 x n_letters> 크기의 원-핫 텐서로 변환
def letter_to_tensor(letter):
    tensor = torch.zeros(1, n_letters)
    tensor[0][letter_to_index(letter)] = 1
    return tensor

# 하나의 이름(line)을 <line_length x 1 x n_letters> 크기의 텐서로 변환
def line_to_tensor(line):
    tensor = torch.zeros(len(line), 1, n_letters)
    for i, letter in enumerate(line):
        tensor[i][0][letter_to_index(letter)] = 1
    return tensor

print("J의 원-핫 텐서:")
print(letter_to_tensor('J'))
print("\nJackson 전체의 텐서 크기:", line_to_tensor('Jackson').size())
print("✅ 데이터 전처리 및 텐서 변환 함수 준비 완료!")

# ------------------------------------------------------------------------------
# 3. RNN 모델 정의
# ------------------------------------------------------------------------------
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(RNN, self).__init__()
        
        self.hidden_size = hidden_size
        
        # 입력과 은닉 상태를 결합하여 다음 은닉 상태와 출력을 만듭니다.
        self.i2h = nn.Linear(input_size + hidden_size, hidden_size)
        self.i2o = nn.Linear(input_size + hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, input_tensor, hidden_tensor):
        combined = torch.cat((input_tensor, hidden_tensor), 1)
        hidden = self.i2h(combined)
        output = self.i2o(combined)
        output = self.softmax(output)
        return output, hidden

    def initHidden(self):
        # 학습 시작 시, 첫 은닉 상태는 0으로 초기화합니다.
        return torch.zeros(1, self.hidden_size)

print("🧠 RNN 모델 클래스 정의 완료!")


# ------------------------------------------------------------------------------
# 4. 학습 준비 (헬퍼 함수, 모델 인스턴스 생성)
# ------------------------------------------------------------------------------

# 모델의 출력(확률)에서 가장 가능성 높은 언어를 찾아내는 함수
def category_from_output(output):
    top_n, top_i = output.topk(1)
    category_i = top_i[0].item()
    return all_categories[category_i], category_i

# 학습에 사용할 랜덤 데이터 쌍을 생성하는 함수
def random_training_example():
    category = random.choice(all_categories)
    line = random.choice(category_lines[category])
    category_tensor = torch.tensor([all_categories.index(category)], dtype=torch.long)
    line_tensor = line_to_tensor(line)
    return category, line, category_tensor, line_tensor

# 하이퍼파라미터 및 모델 관련 설정
n_hidden = 128
learning_rate = 0.005

rnn = RNN(n_letters, n_hidden, n_categories)
criterion = nn.NLLLoss()
optimizer = torch.optim.SGD(rnn.parameters(), lr=learning_rate)

print("🛠️  학습 관련 설정 완료!")

# ------------------------------------------------------------------------------
# 5. 모델 학습
# ------------------------------------------------------------------------------

# 한 스텝 학습을 수행하는 함수
def train(category_tensor, line_tensor):
    hidden = rnn.initHidden()
    optimizer.zero_grad()
    
    for i in range(line_tensor.size()[0]):
        output, hidden = rnn(line_tensor[i], hidden)
    
    loss = criterion(output, category_tensor)
    loss.backward()
    optimizer.step()
    
    return output, loss.item()

# 전체 학습 루프
n_iters = 100000
print_every = 5000
plot_every = 1000

current_loss = 0
all_losses = []

def time_since(since):
    now = time.time()
    s = now - since
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

start = time.time()
print("\n🚀 모델 학습을 시작합니다... (약 2~3분 소요)")

for iter in range(1, n_iters + 1):
    category, line, category_tensor, line_tensor = random_training_example()
    output, loss = train(category_tensor, line_tensor)
    current_loss += loss

    if iter % print_every == 0:
        guess, guess_i = category_from_output(output)
        correct = '✓' if guess == category else f'✗ ({category})'
        print(f'{iter} {iter / n_iters * 100:.0f}% ({time_since(start)}) {loss:.4f} {line} / {guess} {correct}')

    if iter % plot_every == 0:
        all_losses.append(current_loss / plot_every)
        current_loss = 0

print("✅ 모델 학습 완료!")

# 학습 손실 시각화
plt.figure()
plt.plot(all_losses)
plt.title('Training Loss')
plt.xlabel('Iterations (x1000)')
plt.ylabel('Loss')
plt.show()

# ------------------------------------------------------------------------------
# 6. 모델 평가 (혼동 행렬)
# ------------------------------------------------------------------------------
print("\n📊 모델 성능을 평가합니다 (혼동 행렬 생성)...")

# 평가를 위한 헬퍼 함수 (학습 과정 없음)
def evaluate(line_tensor):
    hidden = rnn.initHidden()
    for i in range(line_tensor.size()[0]):
        output, hidden = rnn(line_tensor[i], hidden)
    return output

# 혼동 행렬 생성
with torch.no_grad():
    confusion = torch.zeros(n_categories, n_categories)
    n_confusion = 10000

    for i in range(n_confusion):
        category, line, category_tensor, line_tensor = random_training_example()
        output = evaluate(line_tensor)
        guess, guess_i = category_from_output(output)
        category_i = all_categories.index(category)
        confusion[category_i][guess_i] += 1

    # Normalize by dividing every row by its sum
    for i in range(n_categories):
        confusion[i] = confusion[i] / confusion[i].sum()

    # Set up plot
    fig = plt.figure(figsize=(8,8))
    ax = fig.add_subplot(111)
    cax = ax.matshow(confusion.numpy())
    fig.colorbar(cax)

    # Set up axes
    ax.set_xticklabels([''] + all_categories, rotation=90)
    ax.set_yticklabels([''] + all_categories)

    # Force label at every tick
    ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(1))

    # sphinx_gallery_thumbnail_number = 2
    plt.show()

# ------------------------------------------------------------------------------
# 7. Predicting with a new name
# ------------------------------------------------------------------------------

def predict(input_line, n_predictions=3):
    print(f'\n> {input_line}')
    with torch.no_grad():
        output = evaluate(line_to_tensor(input_line))

        # Get top N categories
        topv, topi = output.topk(n_predictions, 1, True)
        
        for i in range(n_predictions):
            value = torch.exp(topv[0][i]).item() # Convert LogSoftmax output to probability
            category_index = topi[0][i].item()
            print(f'({value:.2%}) {all_categories[category_index]}')

print("\n✨ Let's predict the nationality with a new name.")
predict('Sonesingdara') # Example of a Laotian name
predict('Kim')         # Example of a Korean name
predict('Jackson')     # Example of an English name
predict('Satoshi')     # Example of a Japanese name
predict('Schneider')   # Example of a